# Lexical Representations: Bag-of-Words and TF-IDF

This notebook applies two lexical representation methods to the
adapted conversational corpus introduced in Notebook 1.

Each conversational excerpt is treated as an individual document,
producing a corpus of 20 documents.

The analysis compares raw term frequencies (Bag-of-Words) with
term frequency–inverse document frequency (TF-IDF) weighting.

The objective is to examine how these representations capture
lexical patterns in a small conversational corpus and identify
their limitations.

## 1. Corpus loading

The analysis uses the adapted conversational corpus documented in
Notebook 1. Each excerpt is treated as an individual document
for lexical vectorisation.

In [1]:
import json
from pathlib import Path

import pandas as pd

corpus_path = Path("../data/public/conversations_adapted.json")

with corpus_path.open("r", encoding="utf-8") as f:
    corpus = json.load(f)

In [2]:
type(corpus)

list

In [3]:
corpus[0]

{'exchange_id': 'EX001',
 'title': 'To frankenstein',
 'source_type': 'adapted_public',
 'turns': [{'speaker': 'user',
   'text': "The combined model didn't work, so we sliced the data until something did. We actually pruned a lot of it, made Russian dolls with it, but then we decided to Frankenstein it. And it worked, and it was epic!\n\nOK, I need lunch and a shower, amigo. But I'm going to leave you with a challenging question. Now that we know so much about this, what do you think happened to participants who decided not to choose an option? I wonder whether their brains simply exploded in the middle of a subjunctive ocean!"},
  {'speaker': 'assistant',
   'text': 'HAHAHA. Go eat and shower, because you have just casually left another body on the laboratory floor. 😂\n\nAnd yes, I think those missing responses have suddenly become much more interesting than they looked when they were merely NaN.'}]}

In [4]:
# Extract documents

documents = []

for exchange in corpus:
    for turn in exchange["turns"]:
        documents.append({
            "exchange_id": exchange["exchange_id"],
            "speaker": turn["speaker"],
            "text": turn["text"]
        })

df_documents = pd.DataFrame(documents)

df_documents.head()

,exchange_id,speaker,text
0,EX001,user,"The combined model didn't work, so we sliced t..."
1,EX001,assistant,"HAHAHA. Go eat and shower, because you have ju..."
2,EX002,user,I've just finished editing changes in 19 EDA n...
3,EX002,assistant,SHE'S BACK!\n\nYou didn't come back from Greec...
4,EX003,user,Maria Obsesiones' unicorning in environmentall...


In [5]:
df_documents.shape

(20, 3)

## 2. Bag-of-Words representation

Bag-of-Words represents each document through the frequency of its
vocabulary items, without considering word order or grammatical structure.

We use CountVectorizer to construct a document-term matrix from the
20 conversational excerpts.

In [6]:
# Build document-term matrix

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

bow_matrix = vectorizer.fit_transform(df_documents["text"])

bow_matrix.shape

(20, 677)

In [7]:
# Check feature names

feature_names = vectorizer.get_feature_names_out()

feature_names[:30]

array(['00_intro', '07', '1837', '19', '20', '2022', '2026', '35', '39',
       '48', 'about', 'absolutely', 'academic', 'according', 'accuracy',
       'achieve', 'acquire', 'acquisition', 'act', 'actors', 'actual',
       'actually', 'admit', 'after', 'again', 'agency', 'agent', 'ain',
       'aircraft', 'alive'], dtype=object)

In [8]:
# Turn sparse matrix into array

df_bow = pd.DataFrame(
    bow_matrix.toarray(),
    columns= feature_names,
    index= df_documents.index
)

df_bow.iloc[:5, :15]

,00_intro,07,1837,19,20,2022,2026,35,39,48,about,absolutely,academic,according,accuracy
0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [9]:
# Calculate corpus-wide term frequencies

term_frequencies = df_bow.sum(axis=0).sort_values(ascending=False)
term_frequencies.head(20)

the         89
and         48
to          37
it          29
we          21
that        18
you         17
is          16
what        15
of          12
with        12
for         12
this        10
back         9
notebook     9
would        9
have         9
in           8
no           8
my           7
dtype: int64

In [10]:
term_frequencies[
    term_frequencies.index.str.contains("frankenstein", case=False)
]

frankenstein       7
frankensteining    7
dtype: int64

## 3. TF-IDF representation

TF-IDF (Term Frequency–Inverse Document Frequency) weights terms
according to their frequency within a document and their distribution
across the corpus.

Unlike raw Bag-of-Words counts, TF-IDF reduces the relative weight
of terms occurring across many documents.

We apply TF-IDF to the same 20 excerpts using the default settings,
allowing a direct comparison with the Bag-of-Words representation.

In [11]:
# Build teh TF-IDF matrix

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(df_documents["text"])

tfidf_matrix.shape

(20, 677)

In [12]:
# Create the TF-IDF DataFrame

tfidf_features = tfidf_vectorizer.get_feature_names_out()

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_features,
    index=df_documents.index
)

df_tfidf.iloc[:5, :15]

,00_intro,07,1837,19,20,2022,2026,35,39,48,about,absolutely,academic,according,accuracy
0,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.000000,0.0,0.098377,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.19731,0.19731,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.137665,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0


In [13]:
# Compare same terms

target_terms =["frankenstein", "frankensteining"]

print("Bag-of-Words:")
display(df_bow[target_terms].head(10))

print("TF-IDF:")
display(df_tfidf[target_terms].head(10))

Bag-of-Words:


,frankenstein,frankensteining
0,1,0
1,0,0
2,0,0
3,1,0
4,0,1
5,1,0
6,0,1
7,0,2
8,0,0
9,1,0


TF-IDF:


,frankenstein,frankensteining
0,0.072725,0.000000
1,0.000000,0.000000
2,0.000000,0.000000
3,0.080720,0.000000
4,0.000000,0.042695
5,0.086181,0.000000
6,0.000000,0.116536
7,0.000000,0.134500
8,0.000000,0.000000
9,0.056536,0.000000


In [14]:
# BoW document frequencies

target_terms = ["frankenstein", "frankensteining"]

document_frequencies = (df_bow[target_terms] > 0).sum(axis=0)
document_frequencies

frankenstein       7
frankensteining    6
dtype: int64

In [15]:
(df_bow[target_terms].sum(axis=1) > 0).sum()

np.int64(13)

## 4. Interpretation and limitations

### 4.1 Bag-of-Words

The Bag-of-Words representation produced a document-term matrix containing 20 documents and 677 vocabulary features.

The most frequent terms were common function words, including *the* (89 occurrences), *and* (48) and *to* (37). This illustrates how raw term frequencies reflect lexical occurrence rather than the contextual relevance of individual terms.

The target expressions *frankenstein* and *frankensteining* each occurred seven times. However, BoW represented them as separate vocabulary features, without recognising their morphological relationship.

### 4.2 TF-IDF

TF-IDF produced a matrix with the same dimensions (20 × 677), but replaced raw term counts with weighted values.

Although both target expressions occurred seven times, their document frequencies differed:

| Term | Term frequency | Document frequency |
|---|---:|---:|
| frankenstein | 7 | 7 |
| frankensteining | 7 | 6 |

The seven occurrences of *frankensteining* were distributed across six documents because one excerpt contained the expression twice.

This distinction illustrates the role of document frequency in TF-IDF weighting. Terms appearing in fewer documents receive higher inverse document frequency weights, although the final values also depend on term frequency and document-level normalization.

Consequently, even when a term occurs once in two different documents, its normalized TF-IDF weight may differ.

### 4.3 Comparison and limitations

Both methods represent documents through vocabulary features without explicitly modelling word order, syntactic relationships or contextual meaning.

BoW preserves raw term frequencies, whereas TF-IDF adjusts their weights according to their distribution across the corpus.

Neither representation independently identifies the morphological relationship between *frankenstein* and *frankensteining*, nor distinguishes the different linguistic functions documented in Notebook 1.

The analysis is based on a small, deliberately selected corpus of 20 adapted conversational excerpts. Consequently, vocabulary distributions and TF-IDF weights are specific to this collection and should not be interpreted as representative of broader language use.

**Overall, the comparison demonstrates how BoW and TF-IDF provide different numerical representations of the same texts while sharing important limitations in their treatment of linguistic structure and meaning.**